# 11_callback_guardrail

11_callback_guardrail.py — HITL #1: 콜백 가드레일 (동기 차단)

before_tool_callback 으로 *고위험* 인자를 검사. 위험하면 도구를 우회하고
대체 응답을 반환 → 사람 확인 유도.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 이미 이벤트 루프가 돌아 스크립트의 asyncio.run() 이 깨짐 → nest_asyncio 로 중첩 허용
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '11_callback_guardrail.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
11_callback_guardrail.py — HITL #1: 콜백 가드레일 (동기 차단)

before_tool_callback 으로 *고위험* 인자를 검사. 위험하면 도구를 우회하고
대체 응답을 반환 → 사람 확인 유도.
"""
from google.adk.agents import LlmAgent
from google.adk.tools.tool_context import ToolContext

from _adk_common import adk_model, run_once, banner, adk_unavailable


def transfer_money(amount: int, to: str) -> dict:
    """지정 금액을 송금한다 (고위험 행동).

    Args:
        amount: 송금 금액 (원).
        to: 수취인.
    """
    return {"status": "success",
            "message": f"{to} 에게 {amount:,}원 송금 완료"}


def before_tool(tool, args, tool_context: ToolContext):
    """고액 송금은 자동 실행을 막고 사람 확인을 유도한다."""
    if tool.name == "transfer_money":
        amount = args.get("amount", 0)
        if amount >= 1_000_000:
            # None 이 아닌 dict 를 반환하면 *실제 도구 호출을 건너뛰고* 이 결과를 사용
            return {
                "status": "blocked",
                "message": (
                    f"{amount:,}원 송금은 100만원 이상이라 관리자 승인이 필요합니다. "
                    f"사용자에게 안내하세요."
                ),
            }
    return None  # None → 정상 도구 호출 진행


def main() -> None:
    banner("HITL #1 — before_tool_callback 가드레일")

    agent = LlmAgent(
        name="payment_agent",
        model=adk_model(),
        instruction=(
            "사용자가 송금을 요청하면 transfer_money 도구를 사용. "
            "도구가 blocked 를 반환하면 그 메시지를 그대로 사용자에게 안내."
        ),
        tools=[transfer_money],
        before_tool_callback=before_tool,
    )

    cases = [
        ("정상 금액", "이수에게 5만원 송금해줘"),
        ("고액 차단", "이수에게 200만원 송금해줘"),
    ]
    for label, q in cases:
        print(f"\n  [{label}]  user: {q}")
        try:
            reply = run_once(agent, q)
            print(f"  💬 bot: {reply[:200]}")
        except Exception as e:
            print(f"  ⚠ {type(e).__name__}: {str(e)[:120]}")
            adk_unavailable()
            break


if __name__ == "__main__":
    main()


📌 HITL #1 — before_tool_callback 가드레일

  [정상 금액]  user: 이수에게 5만원 송금해줘


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Node execution failed with exception
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Cl

Root node payment_agent failed.
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Client 


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

  ⚠ RateLimitError: litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Provider returned error","code":429,"
⚠️ ADK 실행 실패 — google-adk[extensions] 설치 + OPENROUTER_API_KEY 확인.
